# Section 2 — Feature Engineering (WoE and IV)

# Notebook 02 — Feature Engineering: WoE and IV

## Central Question
Which variables genuinely predict default — and how do we measure and quantify that precisely?

## Why It Matters
Bivariate analysis in Notebook 01 showed us visually which variables relate
to default. But visual inspection is not enough for a regulatory model.
Information Value answers with a rigorous information-theoretic measure.
Weight of Evidence transforms variables into a scale that makes the logistic
regression linearity assumption valid by construction.

## Data
Loaded from ../data/X_train_clean.csv and ../data/X_test_clean.csv
produced by Notebook 01.

In [52]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from optbinning import OptimalBinning

In [53]:
X_train = pd.read_csv('../data/X_train_clean.csv')
X_test = pd.read_csv('../data/X_test_clean.csv')
y_train = pd.read_csv('../data/y_train.csv').squeeze()
y_test = pd.read_csv('../data/y_test.csv').squeeze()

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Train default rate: {y_train.mean():.4f}")
print(f"Test default rate:  {y_test.mean():.4f}")
print(f"\nFeatures:")
for col in X_train.columns:
    print(f"  {col}")

X_train shape: (120000, 11)
X_test shape:  (30000, 11)
Train default rate: 0.0668
Test default rate:  0.0668

Features:
  RevolvingUtilizationOfUnsecuredLines
  age
  NumberOfTime30-59DaysPastDueNotWorse
  MonthlyIncome
  NumberOfOpenCreditLinesAndLoans
  NumberRealEstateLoansOrLines
  NumberOfDependents
  Ever90DaysPastDue
  Ever60_89DaysPastDue
  Income_missing
  DebtRatio_valid


## Step 1 — Identify Binary and Continuous Features

Binary features (only 2 unique values) require manual WoE calculation.
OptimalBinning cannot create meaningful bins for variables with only two values.
All other features use OptimalBinning's optimal binning algorithm.

In [54]:
binary_features = [col for col in X_train.columns
                   if X_train[col].nunique() == 2]
continuous_features = [col for col in X_train.columns
                       if X_train[col].nunique() > 2]

print(f"Continuous features ({len(continuous_features)}):")
for f in continuous_features:
    print(f"  {f} — {X_train[f].nunique()} unique values")

print(f"\nBinary features ({len(binary_features)}):")
for f in binary_features:
    print(f"  {f} — {X_train[f].value_counts().to_dict()}")

Continuous features (8):
  RevolvingUtilizationOfUnsecuredLines — 99781 unique values
  age — 83 unique values
  NumberOfTime30-59DaysPastDueNotWorse — 14 unique values
  MonthlyIncome — 12655 unique values
  NumberOfOpenCreditLinesAndLoans — 58 unique values
  NumberRealEstateLoansOrLines — 28 unique values
  NumberOfDependents — 13 unique values
  DebtRatio_valid — 86548 unique values

Binary features (3):
  Ever90DaysPastDue — {0: 113508, 1: 6492}
  Ever60_89DaysPastDue — {0: 114095, 1: 5905}
  Income_missing — {0: 94474, 1: 25526}


## Step 2 — Calculate IV for All Features

### Mathematical Foundation

For a variable X binned into B bins, the WoE for bin b is:

    WoE_b = ln(Distr_Good_b / Distr_Bad_b)

Where Distr_Good_b is the proportion of non-defaulters in bin b
and Distr_Bad_b is the proportion of defaulters in bin b.

The Information Value aggregates WoE contributions:

    IV = Σ_b (Distr_Good_b - Distr_Bad_b) × WoE_b

IV is formally equivalent to the symmetric KL divergence between
the good and bad distributions — it measures the total information
content of the variable for distinguishing defaulters from non-defaulters.

IV thresholds (empirical rules from credit scoring practice):
- IV < 0.02  : Useless — drop
- 0.02-0.10  : Weak — keep with caution
- 0.10-0.30  : Medium — good predictor
- 0.30-0.50  : Strong — very useful
- IV > 0.50  : Very Strong — investigate for leakage

In [55]:
def calculate_iv_manual(x, y, feature_name, smoothing=0.5):
    """
    Manual WoE and IV calculation.
    Used for binary variables where OptimalBinning fails.
    Also used as fallback for any OptimalBinning failures.
    """
    df = pd.DataFrame({'x': x, 'y': y})
    
    total_good = (df['y'] == 0).sum()
    total_bad = (df['y'] == 1).sum()
    
    grouped = df.groupby('x')['y'].agg(
        total='count', bad='sum'
    ).reset_index()
    grouped.columns = ['bin', 'total', 'bad']
    grouped['good'] = grouped['total'] - grouped['bad']
    
    n_bins = len(grouped)
    grouped['dist_good'] = (grouped['good'] + smoothing) / (total_good + smoothing * n_bins)
    grouped['dist_bad'] = (grouped['bad'] + smoothing) / (total_bad + smoothing * n_bins)
    grouped['woe'] = np.log(grouped['dist_good'] / grouped['dist_bad'])
    grouped['iv_bin'] = (grouped['dist_good'] - grouped['dist_bad']) * grouped['woe']
    
    total_iv = grouped['iv_bin'].sum()
    woe_map = dict(zip(grouped['bin'], grouped['woe']))
    
    return total_iv, woe_map, grouped

In [56]:
y_vals = y_train.values.astype(int)
results = []
fitted_binners = {}

# Continuous features — OptimalBinning
for feature in continuous_features:
    try:
        optb = OptimalBinning(
            name=feature,
            dtype='numerical',
            max_n_bins=10,
            min_bin_size=0.05
        )
        optb.fit(X_train[feature].values, y_vals)
        
        bt = optb.binning_table.build()
        iv = float(bt['IV'].iloc[-1])
        
        results.append({
            'feature': feature,
            'iv': round(iv, 4),
            'method': 'OptimalBinning'
        })
        fitted_binners[feature] = {'type': 'optbinning', 'binner': optb}
        
    except Exception as e:
        iv, woe_map, _ = calculate_iv_manual(
            X_train[feature].values, y_vals, feature
        )
        results.append({
            'feature': feature,
            'iv': round(iv, 4),
            'method': 'Manual (fallback)'
        })
        fitted_binners[feature] = {'type': 'manual', 'woe_map': woe_map}

# Binary features — always manual
for feature in binary_features:
    iv, woe_map, woe_table = calculate_iv_manual(
        X_train[feature].values, y_vals, feature
    )
    results.append({
        'feature': feature,
        'iv': round(iv, 4),
        'method': 'Manual (binary)'
    })
    fitted_binners[feature] = {'type': 'manual', 'woe_map': woe_map}
    print(f"\n[{feature}] WoE table:")
    print(woe_table[['bin', 'total', 'bad', 'woe', 'iv_bin']])

# Summary
iv_df = pd.DataFrame(results).sort_values('iv', ascending=False).reset_index(drop=True)
print("\nIV RANKING:")
print(iv_df.to_string(index=False))


[Ever90DaysPastDue] WoE table:
   bin   total   bad       woe    iv_bin
0    0  113508  5343  0.371649  0.111432
1    1    6492  2678 -2.282580  0.684386

[Ever60_89DaysPastDue] WoE table:
   bin   total   bad       woe    iv_bin
0    0  114095  5908  0.271341  0.062299
1    1    5905  2113 -2.051452  0.471004

[Income_missing] WoE table:
   bin  total   bad       woe    iv_bin
0    0  94474  6634 -0.052893  0.002254
1    1  25526  1387  0.220213  0.009383

IV RANKING:
                             feature     iv          method
RevolvingUtilizationOfUnsecuredLines 1.1123  OptimalBinning
                   Ever90DaysPastDue 0.7958 Manual (binary)
NumberOfTime30-59DaysPastDueNotWorse 0.6786  OptimalBinning
                Ever60_89DaysPastDue 0.5333 Manual (binary)
                                 age 0.2512  OptimalBinning
     NumberOfOpenCreditLinesAndLoans 0.0870  OptimalBinning
                     DebtRatio_valid 0.0814  OptimalBinning
                       MonthlyIncome 0.0784  

## Step 3 — WoE Transformation

We now transform all features from their raw values to WoE values.

The transformation is fitted on the training set only.
The same WoE mapping is applied to the test set using training-derived boundaries.
This prevents any information from the test set influencing the transformation.

For continuous features we extract bin boundaries directly from the
OptimalBinning table since optb.transform has a version compatibility
issue in this environment.

In [57]:
def extract_woe_from_binning_table(optb):
    """
    Extract WoE boundaries and values from OptimalBinning binning table.
    Parses bin interval strings directly.
    """
    bt = optb.binning_table.build()
    
    # Remove non-bin rows AND rows with empty WoE
    real_bins = bt[
        (~bt['Bin'].isin(['Special', 'Missing', 'Totals'])) &
        (bt['WoE'].astype(str).str.strip() != '') &
        (bt['WoE'].astype(str) != 'nan')
    ].copy()
    
    # Convert WoE to numeric — drop any that fail
    real_bins['WoE_numeric'] = pd.to_numeric(real_bins['WoE'], errors='coerce')
    real_bins = real_bins.dropna(subset=['WoE_numeric'])
    
    boundaries = [-np.inf]
    woe_values = []
    
    for _, row in real_bins.iterrows():
        bin_str = str(row['Bin'])
        woe_val = float(row['WoE_numeric'])
        woe_values.append(woe_val)
        
        # Extract upper boundary from interval string
        numbers = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', bin_str)
        
        if 'inf' in bin_str.lower() and len(numbers) == 0:
            continue
        elif len(numbers) >= 2:
            boundaries.append(float(numbers[-1]))
        elif len(numbers) == 1:
            if 'inf' in bin_str.split(',')[0].lower():
                boundaries.append(float(numbers[0]))
    
    if boundaries[-1] != np.inf:
        boundaries.append(np.inf)
    
    # Verify boundary count matches WoE count
    if len(boundaries) - 1 != len(woe_values):
        print(f"  Boundary count {len(boundaries)-1} != WoE count {len(woe_values)}")
        print(f"  Bins: {real_bins['Bin'].tolist()}")
        print(f"  Boundaries extracted: {boundaries}")
        # Trim to shorter length to avoid index errors
        min_len = min(len(boundaries) - 1, len(woe_values))
        woe_values = woe_values[:min_len]
        boundaries = boundaries[:min_len + 1]
        if boundaries[-1] != np.inf:
            boundaries.append(np.inf)
    
    missing_woe = 0.0
    if 'Missing' in bt['Bin'].values:
        missing_woe_val = pd.to_numeric(
            bt.loc[bt['Bin'] == 'Missing', 'WoE'].iloc[0], errors='coerce'
        )
        if not pd.isna(missing_woe_val):
            missing_woe = float(missing_woe_val)
    
    return boundaries, woe_values, missing_woe


def apply_woe_boundaries(series, boundaries, woe_values, missing_woe=0.0):
    """Apply WoE transformation using explicit bin boundaries."""
    values = np.array(series, dtype=float)
    result = np.full(len(values), missing_woe)
    
    for j in range(len(boundaries) - 1):
        mask = (~np.isnan(values)) & (values >= boundaries[j]) & (values < boundaries[j+1])
        result[mask] = woe_values[j]
    
    return result


# Build WoE lookup for all features
woe_lookup = {}

for feature, binner_info in fitted_binners.items():
    
    if binner_info['type'] == 'optbinning':
        optb = binner_info['binner']
        boundaries, woe_values, missing_woe = extract_woe_from_binning_table(optb)
        woe_lookup[feature] = {
            'type': 'boundaries',
            'boundaries': boundaries,
            'woe_values': woe_values,
            'missing_woe': missing_woe
        }
        print(f"[{feature}] {len(woe_values)} bins | boundaries: {[round(b,3) for b in boundaries]}")
        
    elif binner_info['type'] == 'manual':
        woe_lookup[feature] = {
            'type': 'map',
            'woe_map': binner_info['woe_map']
        }
        print(f"[{feature}] manual map: {binner_info['woe_map']}")

[RevolvingUtilizationOfUnsecuredLines] 9 bins | boundaries: [-inf, 0.06, 0.13, 0.22, 0.3, 0.39, 0.49, 0.7, 0.94, inf]
[age] 10 bins | boundaries: [-inf, 29.5, 36.5, 42.5, 52.5, 55.5, 59.5, 63.5, 67.5, 74.5, inf]
[NumberOfTime30-59DaysPastDueNotWorse] 3 bins | boundaries: [-inf, 0.5, 1.5, inf]
[MonthlyIncome] 9 bins | boundaries: [-inf, 7.42, 7.86, 8.11, 8.48, 8.58, 8.81, 8.98, 9.24, inf]
[NumberOfOpenCreditLinesAndLoans] 8 bins | boundaries: [-inf, 2.5, 3.5, 4.5, 5.5, 7.5, 8.5, 13.5, inf]
[NumberRealEstateLoansOrLines] 4 bins | boundaries: [-inf, 0.5, 1.5, 2.5, inf]
[NumberOfDependents] 4 bins | boundaries: [-inf, 0.5, 1.5, 2.5, inf]
[DebtRatio_valid] 8 bins | boundaries: [-inf, 0.14, 0.23, 0.27, 0.35, 0.42, 0.51, 0.74, inf]
[Ever90DaysPastDue] manual map: {0: 0.371649076037822, 1: -2.282579858363617}
[Ever60_89DaysPastDue] manual map: {0: 0.2713413242666088, 1: -2.051452425357043}
[Income_missing] manual map: {0: -0.0528931536546426, 1: 0.22021341973318342}


In [58]:
X_train_woe = X_train.copy()
X_test_woe = X_test.copy()

for feature, info in woe_lookup.items():
    
    if info['type'] == 'boundaries':
        X_train_woe[feature] = apply_woe_boundaries(
            X_train[feature].values,
            info['boundaries'],
            info['woe_values'],
            info['missing_woe']
        )
        X_test_woe[feature] = apply_woe_boundaries(
            X_test[feature].values,
            info['boundaries'],
            info['woe_values'],
            info['missing_woe']
        )
        
    elif info['type'] == 'map':
        X_train_woe[feature] = X_train[feature].map(info['woe_map']).fillna(0.0)
        X_test_woe[feature] = X_test[feature].map(info['woe_map']).fillna(0.0)

print("WoE transformation complete")
print(f"X_train_woe shape: {X_train_woe.shape}")
print(f"X_test_woe shape:  {X_test_woe.shape}")
print(f"Missing in train:  {X_train_woe.isna().sum().sum()}")
print(f"Missing in test:   {X_test_woe.isna().sum().sum()}")

print(f"\nUnique WoE values per feature:")
for col in X_train_woe.columns:
    n = X_train_woe[col].nunique()
    status = "OK" if n > 1 else "WARNING"
    print(f"  {col}: {n} unique values | {status}")

WoE transformation complete
X_train_woe shape: (120000, 11)
X_test_woe shape:  (30000, 11)
Missing in train:  0
Missing in test:   0

Unique WoE values per feature:
  RevolvingUtilizationOfUnsecuredLines: 9 unique values | OK
  age: 10 unique values | OK
  NumberOfTime30-59DaysPastDueNotWorse: 3 unique values | OK
  MonthlyIncome: 9 unique values | OK
  NumberOfOpenCreditLinesAndLoans: 8 unique values | OK
  NumberRealEstateLoansOrLines: 4 unique values | OK
  NumberOfDependents: 4 unique values | OK
  Ever90DaysPastDue: 2 unique values | OK
  Ever60_89DaysPastDue: 2 unique values | OK
  Income_missing: 2 unique values | OK
  DebtRatio_valid: 8 unique values | OK


In [59]:
X_train_woe.to_csv('../data/X_train_woe.csv', index=False)
X_test_woe.to_csv('../data/X_test_woe.csv', index=False)

print("WoE datasets saved")
print(f"X_train_woe: {X_train_woe.shape}")
print(f"X_test_woe:  {X_test_woe.shape}")

WoE datasets saved
X_train_woe: (120000, 11)
X_test_woe:  (30000, 11)


In [60]:
print("X_train columns:")
print(X_train.columns.tolist())

print("\nX_train_woe columns:")
print(X_train_woe.columns.tolist())

# Check the problematic variable
col = 'NumberOfTime60-89DaysPastDueNotWorse'
if col in X_train.columns:
    print(f"\n{col} value counts:")
    print(X_train[col].value_counts().sort_index())
    print(f"Unique values: {X_train[col].nunique()}")

X_train columns:
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'NumberOfDependents', 'Ever90DaysPastDue', 'Ever60_89DaysPastDue', 'Income_missing', 'DebtRatio_valid']

X_train_woe columns:
['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberRealEstateLoansOrLines', 'NumberOfDependents', 'Ever90DaysPastDue', 'Ever60_89DaysPastDue', 'Income_missing', 'DebtRatio_valid']
